In [9]:
from omegaconf import OmegaConf
from omegaconf.resolvers import oc
import typing as t
from omegaconf import Container
from omegaconf._utils import _DEFAULT_MARKER_
from contextvars import ContextVar
import contextlib

_active_sources: ContextVar[t.Set[str]] = ContextVar('active_sources', default=None)

@contextlib.contextmanager
def capture_sources():
    sources = set()
    token = _active_sources.set(sources)
    try:
        yield sources
    finally:
        _active_sources.reset(token)


def resolve_parameter(
    source: str,
    key: str,
    default: t.Any = _DEFAULT_MARKER_,
    *,
    _parent_: Container,
) -> t.Any:
    from omegaconf._impl import select_value
    active_sources = _active_sources.get()
    if active_sources is not None:
        active_sources.add(source)

    return select_value(cfg=_parent_, key=key, absolute_key=True, default=default)


OmegaConf.register_new_resolver(
    "param",
    resolve_parameter,
    use_cache=False,
)

ValueError: resolver 'param' is already registered

In [ ]:
from omegaconf import OmegaConf
from abc import ABC, abstractmethod
import typing as t
from omegaconf import Container
from omegaconf._utils import _DEFAULT_MARKER_
from contextvars import ContextVar
import contextlib

class VersionedValue(t.NamedTuple):
    version: t.Any
    value: t.Any

# note i was considering making async versions of these but omegaconf is sync
# and it fetches values one value at a time so it wouldnt make sense to make these async
# the only way to get a speed up async is to look the first time the values get fetched at what are all the
# keys that get fetched and then fetch them all in parallel and then cache them inside the ParameterSourceProvider

class ParameterSource(ABC):
    @abstractmethod
    def requires_refresh(self, curr_version:t.Any) -> bool:
        ...

    @abstractmethod
    def get(self, key: str, requested_version: t.Any = _DEFAULT_MARKER_) -> t.Any:
        ...


class ParameterSourceCache(ABC):
    @abstractmethod
    def requires_refresh(self, requested_version: t.Any = _DEFAULT_MARKER_) -> bool:
        ...

    @abstractmethod
    def get(self, key: str, requested_version: t.Any = _DEFAULT_MARKER_) -> t.Any:
        ...

class DefaultParameterSourceCache(ParameterSourceCache):
    def __init__(self, parameter_source: ParameterSource):
        self.parameter_source = parameter_source
        self.cache = {}
        self.curr_version = _DEFAULT_MARKER_

    def requires_refresh(self, requested_version: t.Any = _DEFAULT_MARKER_) -> bool:
        if requested_version is not _DEFAULT_MARKER_ and requested_version != self.curr_version:
            return True
        if self.curr_version is _DEFAULT_MARKER_:
            return True
        return self.parameter_source.requires_refresh(self.curr_version)
    
    def get(self, key: str, requested_version: t.Any = _DEFAULT_MARKER_) -> t.Any:
        if not self.requires_refresh(requested_version=requested_version) and key in self.cache:
            return self.cache[key]
        value, cache_key = self.parameter_source.get(key, requested_version=requested_version)
        if cache_key != self.curr_version:
            self.curr_version = cache_key
            self.cache = {}
        self.cache[key] = value
        return value


class ParameterSourceProvider:
    def __init__(self, sources: t.Dict[str, t.Union[ParameterSource, ParameterSourceCache]]):
        self.sources = {
            name: (source if isinstance(source, ParameterSourceCache) else DefaultParameterSourceCache(source))
            for name, source in sources.items()
        }

    def with_requested_versions(self, requested_versions: t.Dict[str, t.Any]) -> 'ParameterSourceProvider':
        return VersionedParameterSourceProvider(sources=self.sources, versions=requested_versions)
    
    def requires_refresh(self, **kwargs) -> bool:
        return any(source.requires_refresh(**kwargs) for source in self.sources.values())

    def resolve(self, source_name: str, key: str, **kwargs) -> t.Any:
        if source_name not in self.sources:
            raise ValueError(f"Source '{source_name}' not found in provider.")
        return self.sources[source_name].get(key, **kwargs)

class ParameterSourceProviderWithRequest():
    def __init__(self, parameter_source: ParameterSource, request: t.Any):
        self.parameter_source = parameter_source
        self.request = request

    def requires_refresh(self, **kwargs) -> bool:
        return self.parameter_source.requires_refresh(
            request = self.request,
            **kwargs
        )

    def resolve(self, source_name: str, key: str, **kwargs) -> t.Any:
        return self.parameter_source.get(
            source_name=source_name,
            key=key, 
            request=self.request, 
            **kwargs
        )

class ParameterSourceProviderWithVersions():
    def __init__(self, parameter_source: ParameterSource, request: t.Any):        
        super().__init__(sources)
        self.versions = versions

class VersionedParameterSourceProvider(ParameterSourceProvider):
    def __init__(self, sources: t.Dict[str, t.Union[ParameterSource, ParameterSourceCache]], versions: t.Dict[str, t.Any]):
        super().__init__(sources)
        self.versions = versions

_config_source_provider: ContextVar[ParameterSourceProvider] = ContextVar('config_source_provider', default=None)

@contextlib.contextmanager
def config_source_provider(
    provider: ParameterSourceProvider
):
    token = _config_source_provider.set(provider)
    try:
        yield
    finally:
        _config_source_provider.reset(token)


def resolve_parameter(
    source: str,
    key: str,
    default: t.Any = _DEFAULT_MARKER_,
    *,
    _parent_: Container,
) -> t.Any:
    _config_source_provider_instance = _config_source_provider.get()
    if _config_source_provider_instance is not None:
        return _config_source_provider_instance.resolve(source_name=source, key=key)
    if default is _DEFAULT_MARKER_:
        ## TODO should we consider checking if we can resolve static parameters here from the global context
        # I think it might be overkill as there should be no cases we are here and there is no resolver instance
        raise ValueError(f"Parameter source '{source}' not found and no default value provided.")
    return default

OmegaConf.register_new_resolver(
    "param",
    resolve_parameter,
    use_cache=False,
)

In [2]:

    
conf = OmegaConf.create({
    "nested": {
        "value": "${param:test_source,some_key}"
    }
})

In [4]:
with config_source_provider(ParameterSourceProvider(sources={"test_source": ParameterSource("test_source")})):
    resolved = OmegaConf.to_object(conf)
resolved

{'nested': {'value': 'some_key'}}

In [2]:
from pydantic import BaseModel, ConfigDict, model_validator


_providers = {}

class ProviderConfig(BaseModel):
    model_config = ConfigDict(extra="allow")
    type: str

    @model_validator(mode="after")
    def build_provider(self):
        return build_provider(self)

def build_provider(config: ProviderConfig):
    provider_type = config.type
    if provider_type not in _providers:
        raise ValueError(f"Provider type '{provider_type}' not registered.")
    provider_class = _providers[provider_type]
    return provider_class(**config.model_dump())

def register_provider(name, provider):
    global _providers
    _providers[name] = provider


import typing as t
class StaticConfigProvider(BaseModel):
    type: t.Literal["static"] = "static"
    values: t.List[str]

    def test(self):
        print("yebo")

register_provider("static", StaticConfigProvider)

prov = ProviderConfig(type="static", values=["1","2"])
prov

/var/folders/mf/t5j3lfmd2wb_bzv520_hq19h0000gn/T/ipykernel_73629/3326645769.py:36: UserWarning: A custom validator is returning a value other than `self`.
Returning anything other than `self` from a top level model validator isn't supported when validating via `__init__`.
See the `model_validator` docs (https://docs.pydantic.dev/latest/concepts/validators/#model-validators) for more details.
  prov = ProviderConfig(type="static", values=["1","2"])


ProviderConfig(type='static', values=['1', '2'])

In [1]:
from pydantic import BaseModel, RootModel, ConfigDict, model_validator, Field



class ProviderConfig(RootModel):
    root: "_TProvider"


_TProviderUnion = None
_TProvider = None

def register_provider(provider):
    global _TProviderUnion, _TProvider
    if _TProviderUnion is None: _TProviderUnion = provider
    _TProviderUnion = t.Union[_TProviderUnion, provider]
    _TProvider = t.Annotated[_TProviderUnion, Field(discriminator="type")]
    ProviderConfig.model_rebuild()


import typing as t
class StaticConfigProvider(BaseModel):
    type: t.Literal["static"] = "static"
    values: t.List[str]

    def test(self):
        print("yebo")

register_provider(StaticConfigProvider)

prov = ProviderConfig(type="static", values=["1","2"])
prov

ProviderConfig(root=StaticConfigProvider(type='static', values=['1', '2']))

In [5]:
_TProvider

typing.Annotated[__main__.StaticConfigProvider, FieldInfo(annotation=NoneType, required=True, discriminator='type')]